In [1]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import wandb

ENTITY = "ml-for-data-analytics-project"
PROJECT = "energy-forecasting"
PREDICTIONS_KEY = "predictions"

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")
print(f"Found {len(runs)} runs in {ENTITY}/{PROJECT}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/tomasz/.netrc.


Found 18 runs in ml-for-data-analytics-project/energy-forecasting


In [2]:
def _extract_table_path_from_summary(summary_value):
    if isinstance(summary_value, dict):
        path = summary_value.get("path")
        if isinstance(path, str) and path.endswith(".table.json"):
            return path
    return None


def _download_predictions_table(run, key=PREDICTIONS_KEY):
    candidate_paths = []
    summary_path = _extract_table_path_from_summary(run.summary.get(key))
    if summary_path:
        candidate_paths.append(summary_path)

    try:
        for f in run.files():
            name = getattr(f, "name", "")
            if name.endswith(".table.json") and key.lower() in name.lower():
                candidate_paths.append(name)
    except Exception:
        pass

    seen = set()
    for rel_path in candidate_paths:
        if rel_path in seen:
            continue
        seen.add(rel_path)

        try:
            downloaded = run.file(rel_path).download(
                root=tempfile.gettempdir(),
                replace=True,
            )
            with open(downloaded.name, "r", encoding="utf-8") as fp:
                payload = json.load(fp)

            if isinstance(payload, dict) and "columns" in payload and "data" in payload:
                df = pd.DataFrame(payload["data"], columns=payload["columns"])
                return df, rel_path
        except Exception:
            continue

    return None, None


all_predictions = []
run_level_stats = []
missing_predictions = []

for run in runs:
    df_pred, source_path = _download_predictions_table(run)
    if df_pred is None or df_pred.empty:
        missing_predictions.append(run.id)
        continue

    df_pred = df_pred.copy()
    df_pred["run_id"] = run.id
    df_pred["run_name"] = run.name
    df_pred["run_state"] = run.state
    df_pred["table_path"] = source_path

    all_predictions.append(df_pred)

    actual_col = "actual_kWh" if "actual_kWh" in df_pred.columns else None
    pred_col = "predicted_kWh" if "predicted_kWh" in df_pred.columns else None

    if actual_col and pred_col:
        y_true = pd.to_numeric(df_pred[actual_col], errors="coerce")
        y_pred = pd.to_numeric(df_pred[pred_col], errors="coerce")

        mae = float(np.mean(np.abs(y_true - y_pred)))
        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        non_zero = y_true != 0
        mape = (
            float(
                np.mean(
                    np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])
                )
                * 100
            )
            if bool(non_zero.any())
            else np.nan
        )
    else:
        mae = np.nan
        rmse = np.nan
        mape = np.nan

    run_level_stats.append(
        {
            "run_id": run.id,
            "run_name": run.name,
            "state": run.state,
            "created_at": run.created_at,
            "table_rows": int(len(df_pred)),
            "table_cols": int(df_pred.shape[1]),
            "table_path": source_path,
            "summary_val/mae": run.summary.get("val/mae"),
            "summary_val/rmse": run.summary.get("val/rmse"),
            "summary_val/mse": run.summary.get("val/mse"),
            "table_mae": mae,
            "table_rmse": rmse,
            "table_mape_pct": mape,
        }
    )

print(f"Runs with prediction tables: {len(run_level_stats)}")
print(f"Runs without prediction tables: {len(missing_predictions)}")

Runs with prediction tables: 17
Runs without prediction tables: 1


In [3]:
df_pred

,date,actual_kWh,predicted_kWh,error,run_id,run_name,run_state,table_path
0,2017-01-01T00:00:00,30.293125,29.058129,1.234996,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
1,2017-01-02T00:00:00,25.402792,26.651309,-1.248518,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
2,2017-01-03T00:00:00,29.077583,24.106322,4.971261,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
3,2017-01-04T00:00:00,36.197583,29.097314,7.100269,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
4,2017-01-05T00:00:00,36.335583,33.220991,3.114592,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
...,...,...,...,...,...,...,...,...
60,2017-03-02T00:00:00,15.813333,16.775531,-0.962198,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
61,2017-03-03T00:00:00,13.449167,15.146433,-1.697266,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
62,2017-03-04T00:00:00,11.324375,14.581170,-3.256795,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...
63,2017-03-05T00:00:00,13.648833,13.107155,0.541678,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,media/table/predictions_0_5001ef567df28d5c6d0e...


In [4]:
combined_predictions_df = (
    pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
)

prediction_summary_df = (
    pd.DataFrame(run_level_stats)
    .sort_values(
        by="table_rmse",
        ascending=True,
        na_position="last",
    )
    .reset_index(drop=True)
)

combined_path = Path("data/processed/wandb_all_run_predictions.csv")
summary_path = Path("data/processed/wandb_prediction_summary_by_run.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)

if not combined_predictions_df.empty:
    combined_predictions_df.to_csv(combined_path, index=False)
prediction_summary_df.to_csv(summary_path, index=False)

print(f"Saved summary table to: {summary_path}")
if not combined_predictions_df.empty:
    print(f"Saved combined predictions table to: {combined_path}")
if missing_predictions:
    preview_missing = missing_predictions[:10]
    suffix = " ..." if len(missing_predictions) > 10 else ""
    print(f"Missing predictions table in runs: {preview_missing}{suffix}")

prediction_summary_df

Saved summary table to: data/processed/wandb_prediction_summary_by_run.csv
Saved combined predictions table to: data/processed/wandb_all_run_predictions.csv
Missing predictions table in runs: ['pni4b3j8']


,run_id,run_name,state,created_at,table_rows,table_cols,table_path,summary_val/mae,summary_val/rmse,summary_val/mse,table_mae,table_rmse,table_mape_pct
0,vx6nj6rb,"LSTMModel0_lr0.001_seq[1, 2]",finished,2026-03-30T16:19:24Z,63,8,media/table/predictions_49_f034b1cbb669941840c...,0.041127,0.051586,0.002661,1.402818,1.759589,5.382815
1,4pv5mdmd,"LSTMModel0_lr0.001_seq[1, 2]_dropout",finished,2026-03-30T16:27:40Z,63,8,media/table/predictions_49_157afef3f2b36b452ea...,0.049643,0.062289,0.003880,1.693316,2.124662,6.292367
2,i6nfjq31,"DECOMPOSE_ARIMA_order(1, 1, 2)_p7",finished,2026-04-24T22:43:43Z,65,8,media/table/predictions_0_5cb9fff2dfd65cf08277...,2.727446,3.352765,11.241033,2.727446,3.352765,10.976243
3,znxzbbdy,"SARIMA_order(0, 1, 3)_sorder(1, 1, 1, 7)",finished,2026-04-24T22:44:47Z,65,8,media/table/predictions_0_5001ef567df28d5c6d0e...,2.728761,3.410608,11.632247,2.728761,3.410608,11.022812
4,fwg0uzd9,"SARIMAX_order(0, 1, 3)_sorder(0, 1, 1, 7)",finished,2026-04-24T22:03:15Z,65,8,media/table/predictions_0_8db6d0f64cfd5a839e11...,2.725984,3.420405,11.699173,2.725984,3.420405,11.115014
5,pvq2ytfa,encdec_rs_t06_lr0.001_dr0.3_hs64_bs16_ep80,finished,2026-04-24T06:33:29Z,65,8,media/table/predictions_79_0989713a73222252ec1...,0.092165,0.116993,0.013687,3.143694,3.990579,11.199957
6,y48n2ot7,encdec_rs_t07_lr0.0001_dr0.1_hs128_bs16_ep80,finished,2026-04-24T06:33:44Z,65,8,media/table/predictions_79_cfe628bd5b1b23af806...,0.092534,0.118598,0.014065,3.156292,4.045310,11.599917
7,n9psbfq2,encdec_rs_t05_lr0.001_dr0.1_hs64_bs64_ep100,finished,2026-04-24T06:33:13Z,65,8,media/table/predictions_99_61d7d7b7298b038632a...,0.093577,0.119412,0.014259,3.191856,4.073094,11.255251
8,asu5y1ij,encdec_rs_t04_lr0.0001_dr0.2_hs128_bs16_ep100,finished,2026-04-24T06:32:54Z,65,8,media/table/predictions_99_5d455e0061ff97ef21b...,0.091285,0.119587,0.014301,3.113689,4.079043,10.970803
9,e9cczey0,"keras_LSTM_encoder_decoder_lr0.001_seq[1, 2]",finished,2026-04-06T12:24:34Z,65,8,media/table/predictions_49_848054a8cc6d04d06c7...,0.096310,0.122415,0.014985,3.285097,4.175510,11.686592


In [5]:
from datetime import datetime, timezone

# Fallback so this cell can run independently after kernel restart.
if "prediction_summary_df" not in globals() or prediction_summary_df.empty:
    summary_path = Path("data/processed/wandb_prediction_summary_by_run.csv")
    prediction_summary_df = pd.read_csv(summary_path)

if "combined_predictions_df" not in globals() or combined_predictions_df.empty:
    combined_path = Path("data/processed/wandb_all_run_predictions.csv")
    if combined_path.exists():
        combined_predictions_df = pd.read_csv(combined_path)
    else:
        combined_predictions_df = pd.DataFrame()

wandb.login()

upload_run = wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="analysis",
    name=f"prediction-summary-upload-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
)

upload_run.log(
    {
        "prediction_summary_by_run": wandb.Table(dataframe=prediction_summary_df),
    }
)

artifact = wandb.Artifact(
    name="prediction-summary-by-run",
    type="dataset",
    description="Summary and combined prediction tables aggregated from all project runs.",
)
artifact.add_file(str(summary_path))

if "combined_path" in globals() and Path(combined_path).exists():
    artifact.add_file(str(combined_path))

upload_run.log_artifact(artifact)
upload_run.finish()

print(f"Uploaded table to run: {upload_run.url}")
print("Logged table key: prediction_summary_by_run")
print("Logged artifact name: prediction-summary-by-run")

wandb: Currently logged in as: 275987 (ml-for-data-analytics-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Uploaded table to run: https://wandb.ai/ml-for-data-analytics-project/energy-forecasting/runs/yynhasqk
Logged table key: prediction_summary_by_run
Logged artifact name: prediction-summary-by-run
